In [1]:

import pandas as pd
from pathlib import Path

# Define the original and output files
input_path = Path("data") / "netflix_titles.csv"
output_path = Path("data") / "netflix_titles_cleaned.csv"

# Load the original data
df = pd.read_csv(input_path)

# Record the original date whitespace count
before = (
    df["date_added"].notna()
    & df["date_added"].ne(df["date_added"].str.strip())
).sum()

# Trim leading/trailing whitespace from text columns
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    df[col] = df[col].str.strip()

# Verify the result
after = (
    df["date_added"].notna()
    & df["date_added"].ne(df["date_added"].str.strip())
).sum()

print("DATE WHITESPACE CLEANING")
print("Affected values before:", int(before))
print("Affected values after:", int(after))
print("Rows preserved:", len(df))


suspicious_ratings = ["66 min", "74 min", "84 min"]

mask = df["rating"].isin(suspicious_ratings)

df.loc[mask, "duration"] = df.loc[mask, "rating"]
df.loc[mask, "rating"] = pd.NA
# Save to a new file, never overwrite the original
df.to_csv(output_path, index=False)

print("\nSaved cleaned copy to:", output_path)


DATE WHITESPACE CLEANING
Affected values before: 88
Affected values after: 0
Rows preserved: 8807

Saved cleaned copy to: data\netflix_titles_cleaned.csv


In [2]:

# Task 2.12.1: Verify the saved cleaned file

check_df = pd.read_csv(output_path)

print("\nSAVED FILE VERIFICATION")
print("File exists:", output_path.exists())
print("Original shape:", (8807, 12))
print("Saved shape:", check_df.shape)
print("Columns preserved:", list(check_df.columns) == list(df.columns))
print("Missing IDs:", int(check_df["show_id"].isna().sum()))
print("Duplicate IDs:", int(check_df["show_id"].duplicated().sum()))

saved_spaces = (
    check_df["date_added"].notna()
    & check_df["date_added"].ne(check_df["date_added"].str.strip())
).sum()

print("Date whitespace remaining:", int(saved_spaces))



SAVED FILE VERIFICATION
File exists: True
Original shape: (8807, 12)
Saved shape: (8807, 12)
Columns preserved: True
Missing IDs: 0
Duplicate IDs: 0
Date whitespace remaining: 0


In [3]:

# Task 2.13: Recover durations misplaced in rating

suspicious_ratings = ["66 min", "74 min", "84 min"]

mask = df["rating"].isin(suspicious_ratings)

# Show the affected records before correction
print("\nMISPLACED DURATION CORRECTION")
print(
    df.loc[mask, ["show_id", "title", "rating", "duration"]]
    .to_string(index=False)
)

# Recover duration and leave rating unknown
df.loc[mask, "duration"] = df.loc[mask, "rating"]
df.loc[mask, "rating"] = pd.NA

print("\nRecords corrected:", int(mask.sum()))
print("Suspicious rating values remaining:",
      int(df["rating"].isin(suspicious_ratings).sum()))
print("Missing ratings after correction:",
      int(df["rating"].isna().sum()))



MISPLACED DURATION CORRECTION
Empty DataFrame
Columns: [show_id, title, rating, duration]
Index: []

Records corrected: 0
Suspicious rating values remaining: 0
Missing ratings after correction: 7


In [4]:

# Task 2.13.1: Verify corrections in the saved CSV

saved_df = pd.read_csv(output_path)

corrected_titles = [
    "Louis C.K. 2017",
    "Louis C.K.: Hilarious",
    "Louis C.K.: Live at the Comedy Store"
]

saved_records = saved_df[
    saved_df["title"].isin(corrected_titles)
]

print("\nSAVED CORRECTION VERIFICATION")
print(
    saved_records[
        ["show_id", "title", "rating", "duration"]
    ].to_string(index=False)
)

print("\nCorrected records found:", len(saved_records))
print(
    "Corrected ratings missing:",
    int(saved_records["rating"].isna().sum())
)
print(
    "Corrected durations present:",
    int(saved_records["duration"].notna().sum())
)
print(
    "Suspicious ratings remaining:",
    int(saved_df["rating"].isin(
        ["66 min", "74 min", "84 min"]
    ).sum())
)



SAVED CORRECTION VERIFICATION
show_id                                title rating duration
  s5542                      Louis C.K. 2017    NaN   74 min
  s5795                Louis C.K.: Hilarious    NaN   84 min
  s5814 Louis C.K.: Live at the Comedy Store    NaN   66 min

Corrected records found: 3
Corrected ratings missing: 3
Corrected durations present: 3
Suspicious ratings remaining: 0


In [5]:

# Task 2.14: Compare missing values before and after cleaning

original_df = pd.read_csv(input_path)
cleaned_df = pd.read_csv(output_path)

comparison = pd.DataFrame({
    "original_missing": original_df.isna().sum(),
    "cleaned_missing": cleaned_df.isna().sum()
})

comparison["change"] = (
    comparison["original_missing"]
    - comparison["cleaned_missing"]
)

print("\nMISSING-VALUE COMPARISON")
print(comparison.to_string())

original_total = int(original_df.isna().sum().sum())
cleaned_total = int(cleaned_df.isna().sum().sum())

print("\nOriginal missing cells:", original_total)
print("Cleaned missing cells:", cleaned_total)
print(
    "Original missing rate (%):",
    round(original_df.isna().sum().sum() / original_df.size * 100, 2)
)
print(
    "Cleaned missing rate (%):",
    round(cleaned_df.isna().sum().sum() / cleaned_df.size * 100, 2)
)



MISSING-VALUE COMPARISON
              original_missing  cleaned_missing  change
show_id                      0                0       0
type                         0                0       0
title                        0                0       0
director                  2634             2634       0
cast                       825              825       0
country                    831              831       0
date_added                  10               10       0
release_year                 0                0       0
rating                       4                7      -3
duration                     3                0       3
listed_in                    0                0       0
description                  0                0       0

Original missing cells: 4307
Cleaned missing cells: 4307
Original missing rate (%): 4.08
Cleaned missing rate (%): 4.08


In [6]:

# Task 2.15: Final dataset integrity checks

cleaned_df = pd.read_csv(output_path)

print("\nFINAL INTEGRITY CHECKS")

print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))
print("Column names:", cleaned_df.columns.tolist())

print("Missing show_id values:", cleaned_df["show_id"].isna().sum())
print("Duplicate show_id values:", cleaned_df["show_id"].duplicated().sum())

print("\nType counts:")
print(cleaned_df["type"].value_counts(dropna=False))

print("\nUnexpected type values:")
print(
    cleaned_df.loc[
        ~cleaned_df["type"].isin(["Movie", "TV Show"]),
        "type"
    ].value_counts(dropna=False)
)

print("\nRemaining suspicious rating values:")
print(
    cleaned_df[
        cleaned_df["rating"].isin(["66 min", "74 min", "84 min"])
    ][["show_id", "title", "rating", "duration"]]
)

print("\nFinal dataset shape:", cleaned_df.shape)



FINAL INTEGRITY CHECKS
Rows: 8807
Columns: 12
Column names: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']
Missing show_id values: 0
Duplicate show_id values: 0

Type counts:
type
Movie      6131
TV Show    2676
Name: count, dtype: int64

Unexpected type values:
Series([], Name: count, dtype: int64)

Remaining suspicious rating values:
Empty DataFrame
Columns: [show_id, title, rating, duration]
Index: []

Final dataset shape: (8807, 12)


In [7]:

# Task 2.16: Audit date parsing

cleaned_df = pd.read_csv(output_path)

date_text = cleaned_df["date_added"].astype("string").str.strip()
parsed_dates = pd.to_datetime(
    date_text,
    format="%B %d, %Y",
    errors="coerce"
)

missing_dates = date_text.isna().sum()
failed_dates = (date_text.notna() & parsed_dates.isna()).sum()

print("\nDATE PARSING AUDIT")
print("Missing date_added values:", missing_dates)
print("Dates parsed successfully:", parsed_dates.notna().sum())
print("Non-missing dates that failed parsing:", failed_dates)
print("Earliest date:", parsed_dates.min())
print("Latest date:", parsed_dates.max())

print("\nExamples of parsed dates:")
print(parsed_dates.dropna().head(5))



DATE PARSING AUDIT
Missing date_added values: 10
Dates parsed successfully: 8797
Non-missing dates that failed parsing: 0
Earliest date: 2008-01-01 00:00:00
Latest date: 2021-09-25 00:00:00

Examples of parsed dates:
0   2021-09-25
1   2021-09-24
2   2021-09-24
3   2021-09-24
4   2021-09-24
Name: date_added, dtype: datetime64[ns]


In [8]:

# Task 2.17: Create a parsed datetime column

cleaned_df = pd.read_csv(output_path)

cleaned_df["date_added_parsed"] = pd.to_datetime(
    cleaned_df["date_added"].astype("string").str.strip(),
    format="%B %d, %Y",
    errors="coerce"
)

print("\nPARSED DATE COLUMN VERIFICATION")
print("Total rows:", len(cleaned_df))
print(
    "Parsed date column dtype:",
    cleaned_df["date_added_parsed"].dtype
)
print(
    "Non-missing parsed dates:",
    cleaned_df["date_added_parsed"].notna().sum()
)
print(
    "Missing parsed dates:",
    cleaned_df["date_added_parsed"].isna().sum()
)
print(
    "Original date_added column preserved:",
    "date_added" in cleaned_df.columns
)

print("\nSample date comparison:")
print(
    cleaned_df[
        ["show_id", "date_added", "date_added_parsed"]
    ].head(10).to_string(index=False)
)



PARSED DATE COLUMN VERIFICATION
Total rows: 8807
Parsed date column dtype: datetime64[ns]
Non-missing parsed dates: 8797
Missing parsed dates: 10
Original date_added column preserved: True

Sample date comparison:
show_id         date_added date_added_parsed
     s1 September 25, 2021        2021-09-25
     s2 September 24, 2021        2021-09-24
     s3 September 24, 2021        2021-09-24
     s4 September 24, 2021        2021-09-24
     s5 September 24, 2021        2021-09-24
     s6 September 24, 2021        2021-09-24
     s7 September 24, 2021        2021-09-24
     s8 September 24, 2021        2021-09-24
     s9 September 24, 2021        2021-09-24
    s10 September 24, 2021        2021-09-24


In [9]:

# Task 2.18: Save and verify parsed date column

cleaned_df.to_csv(output_path, index=False)

# Reload the saved file to verify persistence
saved_df = pd.read_csv(output_path)

print("\nSAVED DATE COLUMN VERIFICATION")
print("Saved file:", output_path)
print("Rows:", len(saved_df))
print("Columns:", len(saved_df.columns))
print(
    "date_added column preserved:",
    "date_added" in saved_df.columns
)
print(
    "date_added_parsed column saved:",
    "date_added_parsed" in saved_df.columns
)

# CSV stores datetime values as text, so parse them again for analysis
saved_dates = pd.to_datetime(
    saved_df["date_added_parsed"],
    errors="coerce"
)

print("Successfully reloaded dates:", saved_dates.notna().sum())
print("Missing saved dates:", saved_dates.isna().sum())
print("Earliest saved date:", saved_dates.min())
print("Latest saved date:", saved_dates.max())



SAVED DATE COLUMN VERIFICATION
Saved file: data\netflix_titles_cleaned.csv
Rows: 8807
Columns: 13
date_added column preserved: True
date_added_parsed column saved: True
Successfully reloaded dates: 8797
Missing saved dates: 10
Earliest saved date: 2008-01-01 00:00:00
Latest saved date: 2021-09-25 00:00:00


In [10]:

# Task 2.19: Audit duration values

audit_df = pd.read_csv(output_path)

movie_mask = audit_df["type"].eq("Movie")
tv_mask = audit_df["type"].eq("TV Show")

movie_duration_valid = (
    audit_df.loc[movie_mask, "duration"]
    .astype("string")
    .str.fullmatch(r"\d+\s+min", na=False)
)

tv_duration_valid = (
    audit_df.loc[tv_mask, "duration"]
    .astype("string")
    .str.fullmatch(r"\d+\s+Seasons?", na=False)
)

print("\nDURATION AUDIT")
print("Total rows:", len(audit_df))
print("Movies:", int(movie_mask.sum()))
print("TV Shows:", int(tv_mask.sum()))

print("\nMovie duration values")
print("Missing:", int(audit_df.loc[movie_mask, "duration"].isna().sum()))
print("Valid minute formats:", int(movie_duration_valid.sum()))
print("Invalid formats:", int((~movie_duration_valid).sum()))

print("\nTV Show duration values")
print("Missing:", int(audit_df.loc[tv_mask, "duration"].isna().sum()))
print("Valid season formats:", int(tv_duration_valid.sum()))
print("Invalid formats:", int((~tv_duration_valid).sum()))

print("\nExamples:")
print(
    audit_df[["type", "title", "duration"]]
    .dropna(subset=["duration"])
    .head(10)
    .to_string(index=False)
)



DURATION AUDIT
Total rows: 8807
Movies: 6131
TV Shows: 2676

Movie duration values
Missing: 0
Valid minute formats: 6131
Invalid formats: 0

TV Show duration values
Missing: 0
Valid season formats: 2676
Invalid formats: 0

Examples:
   type                            title  duration
  Movie             Dick Johnson Is Dead    90 min
TV Show                    Blood & Water 2 Seasons
TV Show                        Ganglands  1 Season
TV Show            Jailbirds New Orleans  1 Season
TV Show                     Kota Factory 2 Seasons
TV Show                    Midnight Mass  1 Season
  Movie My Little Pony: A New Generation    91 min
  Movie                          Sankofa   125 min
TV Show    The Great British Baking Show 9 Seasons
  Movie                     The Starling   104 min


In [11]:

# Task 2.20: Create numeric duration columns

duration_df = pd.read_csv(output_path)

duration_df["duration_minutes"] = pd.to_numeric(
    duration_df["duration"]
    .where(duration_df["type"].eq("Movie"))
    .str.extract(r"(\d+)", expand=False),
    errors="coerce"
).astype("Int64")

duration_df["duration_seasons"] = pd.to_numeric(
    duration_df["duration"]
    .where(duration_df["type"].eq("TV Show"))
    .str.extract(r"(\d+)", expand=False),
    errors="coerce"
).astype("Int64")

print("\nNUMERIC DURATION VERIFICATION")
print("Total rows:", len(duration_df))

print(
    "Movies with numeric minutes:",
    duration_df.loc[
        duration_df["type"].eq("Movie"),
        "duration_minutes"
    ].notna().sum()
)

print(
    "TV Shows with numeric seasons:",
    duration_df.loc[
        duration_df["type"].eq("TV Show"),
        "duration_seasons"
    ].notna().sum()
)

print(
    "Movies with season values:",
    duration_df.loc[
        duration_df["type"].eq("Movie"),
        "duration_seasons"
    ].notna().sum()
)

print(
    "TV Shows with minute values:",
    duration_df.loc[
        duration_df["type"].eq("TV Show"),
        "duration_minutes"
    ].notna().sum()
)

print("\nSample transformed values:")
print(
    duration_df[
        ["type", "title", "duration",
         "duration_minutes", "duration_seasons"]
    ].head(10).to_string(index=False)
)



NUMERIC DURATION VERIFICATION
Total rows: 8807
Movies with numeric minutes: 6131
TV Shows with numeric seasons: 2676
Movies with season values: 0
TV Shows with minute values: 0

Sample transformed values:
   type                            title  duration  duration_minutes  duration_seasons
  Movie             Dick Johnson Is Dead    90 min                90              <NA>
TV Show                    Blood & Water 2 Seasons              <NA>                 2
TV Show                        Ganglands  1 Season              <NA>                 1
TV Show            Jailbirds New Orleans  1 Season              <NA>                 1
TV Show                     Kota Factory 2 Seasons              <NA>                 2
TV Show                    Midnight Mass  1 Season              <NA>                 1
  Movie My Little Pony: A New Generation    91 min                91              <NA>
  Movie                          Sankofa   125 min               125              <NA>
TV Show    

In [12]:

# Task 2.21: Save and verify numeric duration columns

duration_df.to_csv(output_path, index=False)

saved_df = pd.read_csv(output_path)

print("\nSAVED DURATION COLUMNS VERIFICATION")
print("Saved file:", output_path)
print("Rows:", len(saved_df))
print("Columns:", len(saved_df.columns))

print(
    "duration_minutes column saved:",
    "duration_minutes" in saved_df.columns
)
print(
    "duration_seasons column saved:",
    "duration_seasons" in saved_df.columns
)

print(
    "Movies with numeric minutes:",
    saved_df.loc[
        saved_df["type"].eq("Movie"),
        "duration_minutes"
    ].notna().sum()
)
print(
    "TV Shows with numeric seasons:",
    saved_df.loc[
        saved_df["type"].eq("TV Show"),
        "duration_seasons"
    ].notna().sum()
)

print(
    "Movies with season values:",
    saved_df.loc[
        saved_df["type"].eq("Movie"),
        "duration_seasons"
    ].notna().sum()
)
print(
    "TV Shows with minute values:",
    saved_df.loc[
        saved_df["type"].eq("TV Show"),
        "duration_minutes"
    ].notna().sum()
)

print("\nSaved sample:")
print(
    saved_df[
        ["type", "title", "duration",
         "duration_minutes", "duration_seasons"]
    ].head(10).to_string(index=False)
)



SAVED DURATION COLUMNS VERIFICATION
Saved file: data\netflix_titles_cleaned.csv
Rows: 8807
Columns: 15
duration_minutes column saved: True
duration_seasons column saved: True
Movies with numeric minutes: 6131
TV Shows with numeric seasons: 2676
Movies with season values: 0
TV Shows with minute values: 0

Saved sample:
   type                            title  duration  duration_minutes  duration_seasons
  Movie             Dick Johnson Is Dead    90 min              90.0               NaN
TV Show                    Blood & Water 2 Seasons               NaN               2.0
TV Show                        Ganglands  1 Season               NaN               1.0
TV Show            Jailbirds New Orleans  1 Season               NaN               1.0
TV Show                     Kota Factory 2 Seasons               NaN               2.0
TV Show                    Midnight Mass  1 Season               NaN               1.0
  Movie My Little Pony: A New Generation    91 min              91.0  

In [13]:

# Task 2.22: Final cleaned dataset validation

final_df = pd.read_csv(output_path)

original_columns = [
    "show_id", "type", "title", "director", "cast",
    "country", "date_added", "release_year", "rating",
    "duration", "listed_in", "description"
]

print("\nFINAL CLEANED DATASET VALIDATION")

print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))
print(
    "All original columns preserved:",
    all(col in final_df.columns for col in original_columns)
)
print("Unique IDs:", final_df["show_id"].nunique())
print("Duplicate IDs:", final_df["show_id"].duplicated().sum())

print("\nMissing-value totals by column:")
print(final_df.isna().sum().to_string())

print("\nDuration validation:")
print(
    "Movies missing minutes:",
    final_df.loc[
        final_df["type"].eq("Movie"), "duration_minutes"
    ].isna().sum()
)
print(
    "TV Shows missing seasons:",
    final_df.loc[
        final_df["type"].eq("TV Show"), "duration_seasons"
    ].isna().sum()
)
print(
    "Movies with season values:",
    final_df.loc[
        final_df["type"].eq("Movie"), "duration_seasons"
    ].notna().sum()
)
print(
    "TV Shows with minute values:",
    final_df.loc[
        final_df["type"].eq("TV Show"), "duration_minutes"
    ].notna().sum()
)

print("\nCorrected record verification:")
print(
    final_df.loc[
        final_df["show_id"].isin(["s5542", "s5795", "s5814"]),
        ["show_id", "title", "rating", "duration",
         "duration_minutes", "duration_seasons"]
    ].to_string(index=False)
)



FINAL CLEANED DATASET VALIDATION
Rows: 8807
Columns: 15
All original columns preserved: True
Unique IDs: 8807
Duplicate IDs: 0

Missing-value totals by column:
show_id                 0
type                    0
title                   0
director             2634
cast                  825
country               831
date_added             10
release_year            0
rating                  7
duration                0
listed_in               0
description             0
date_added_parsed      10
duration_minutes     2676
duration_seasons     6131

Duration validation:
Movies missing minutes: 0
TV Shows missing seasons: 0
Movies with season values: 0
TV Shows with minute values: 0

Corrected record verification:
show_id                                title rating duration  duration_minutes  duration_seasons
  s5542                      Louis C.K. 2017    NaN   74 min              74.0               NaN
  s5795                Louis C.K.: Hilarious    NaN   84 min              84.0         

In [14]:

# Task 2.23: Compare original and cleaned datasets

original_df = pd.read_csv(input_path)
cleaned_df = pd.read_csv(output_path)

print("\nORIGINAL VS CLEANED DATASET")

print("Original shape:", original_df.shape)
print("Cleaned shape:", cleaned_df.shape)

print(
    "Original file still has 12 columns:",
    len(original_df.columns) == 12
)

print(
    "Original has no added analysis columns:",
    "date_added_parsed" not in original_df.columns
    and "duration_minutes" not in original_df.columns
    and "duration_seasons" not in original_df.columns
)

print(
    "Original duration values remain missing:",
    original_df["duration"].isna().sum()
)

print(
    "Cleaned duration values missing:",
    cleaned_df["duration"].isna().sum()
)

print(
    "Original suspicious rating values:",
    original_df["rating"].isin(
        ["66 min", "74 min", "84 min"]
    ).sum()
)

print(
    "Cleaned suspicious rating values:",
    cleaned_df["rating"].isin(
        ["66 min", "74 min", "84 min"]
    ).sum()
)



ORIGINAL VS CLEANED DATASET
Original shape: (8807, 12)
Cleaned shape: (8807, 15)
Original file still has 12 columns: True
Original has no added analysis columns: True
Original duration values remain missing: 3
Cleaned duration values missing: 0
Original suspicious rating values: 3
Cleaned suspicious rating values: 0
